# Leakage-free evaluation of hyperspectral classifiers

Reproduces the whole study on one Kaggle GPU session.

**Settings:** turn ON *GPU* and *Internet* in the notebook settings panel. No *Add Input* is
needed: the second cell downloads the four scenes from their public mirrors.

Runtime on a T4 is roughly one hour for the protocol cross and two hours for the
mechanism experiments, so run them as separate sessions if a single session is tight.


In [ ]:
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

In [ ]:

import json, os, re, urllib.request, urllib.parse
os.makedirs("data", exist_ok=True)
JOBS = [
 ("cu4oral6/hsi_datasets_unmixing","main","Pavia_Centre_and_University/raw/PaviaU.mat","PaviaU.mat"),
 ("cu4oral6/hsi_datasets_unmixing","main","Pavia_Centre_and_University/gt/PaviaU_gt.mat","PaviaU_gt.mat"),
 ("cu4oral6/hsi_datasets_unmixing","main","Pavia_Centre_and_University/raw/Pavia.mat","Pavia.mat"),
 ("cu4oral6/hsi_datasets_unmixing","main","Pavia_Centre_and_University/gt/Pavia_gt.mat","Pavia_gt.mat"),
 ("bluemountain1231/multimodal-rs-classification","main","data/houston2013/Houston.mat","Houston.mat"),
 ("bluemountain1231/multimodal-rs-classification","main","data/houston2013/Houston_gt.mat","Houston_gt.mat"),
]
ZENODO = [
 ("https://zenodo.org/api/records/15771735/files/Salinas_corrected.mat/content","Salinas_corrected.mat"),
 ("https://zenodo.org/api/records/15771735/files/Salinas_gt.mat/content","Salinas_gt.mat"),
]

def pointer(repo, br, path):
    u = f"https://raw.githubusercontent.com/{repo}/{br}/{urllib.parse.quote(path)}"
    return urllib.request.urlopen(u, timeout=120).read().decode("utf-8", "ignore")

by_repo = {}
for repo, br, path, out in JOBS:
    if os.path.exists(f"data/{out}"):
        continue
    txt = pointer(repo, br, path)
    oid = re.search(r"oid sha256:([0-9a-f]+)", txt)
    size = re.search(r"size (\d+)", txt)
    if not oid:
        open(f"data/{out}", "wb").write(txt.encode())
        continue
    by_repo.setdefault(repo, []).append(
        {"oid": oid.group(1), "size": int(size.group(1)), "_f": out})

for repo, objs in by_repo.items():
    body = json.dumps({"operation": "download", "transfers": ["basic"],
                       "objects": [{"oid": o["oid"], "size": o["size"]} for o in objs]}).encode()
    rq = urllib.request.Request(
        f"https://github.com/{repo}.git/info/lfs/objects/batch", data=body,
        headers={"Content-Type": "application/vnd.git-lfs+json",
                 "Accept": "application/vnd.git-lfs+json"})
    r = json.loads(urllib.request.urlopen(rq, timeout=180).read())
    m = {o["oid"]: o for o in r["objects"]}
    for o in objs:
        a = m[o["oid"]]["actions"]["download"]
        data = urllib.request.urlopen(
            urllib.request.Request(a["href"], headers=a.get("header", {})), timeout=900).read()
        open(f"data/{o['_f']}", "wb").write(data)
        print(o["_f"], len(data), "bytes", flush=True)

for url, out in ZENODO:
    if os.path.exists(f"data/{out}"):
        continue
    data = urllib.request.urlopen(url, timeout=900).read()
    open(f"data/{out}", "wb").write(data)
    print(out, len(data), "bytes", flush=True)

print(sorted(os.listdir("data")))


In [ ]:
import os
os.makedirs('src', exist_ok=True)
open('src/hsidata.py','w').write('"""Data handling and leakage-free splitting for hyperspectral image classification.\n\nTwo sampling protocols are provided side by side so the same model can be\nevaluated under both:\n\n``random``    the conventional protocol.  Labelled pixels are drawn at random,\n              so a test pixel\'s patch window routinely overlaps the windows of\n              training pixels.\n\n``disjoint``  spatial blocks are assigned whole to train / val / test, and any\n              labelled pixel whose patch window reaches into a block belonging\n              to another split is discarded.  The discarded ring acts as a\n              buffer, so no test patch can contain a training pixel.\n\nBand standardisation is always fitted on training pixels only.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nfrom pathlib import Path\n\nimport numpy as np\nimport scipy.io as sio\nfrom scipy import ndimage\n\n# Scene name -> (cube file, ground-truth file).\nDATASETS: dict[str, tuple[str, str]] = {\n    "IndianPines": ("Indian_pines_corrected.mat", "Indian_pines_gt.mat"),\n    "PaviaU": ("PaviaU.mat", "PaviaU_gt.mat"),\n    "Salinas": ("Salinas_corrected.mat", "Salinas_gt.mat"),\n    "PaviaC": ("Pavia.mat", "Pavia_gt.mat"),\n    "Botswana": ("Botswana.mat", "Botswana_gt.mat"),\n    "KSC": ("KSC.mat", "KSC_gt.mat"),\n    "Houston": ("Houston.mat", "Houston_gt.mat"),\n}\n\nSPLIT_NAMES = ("train", "val", "test")\n\n\ndef _first_array(mat: dict) -> np.ndarray:\n    keys = [k for k in mat if not k.startswith("__")]\n    if len(keys) != 1:\n        keys = sorted(keys, key=lambda k: np.asarray(mat[k]).size, reverse=True)\n    return np.asarray(mat[keys[0]])\n\n\ndef load_cube(name: str, root: str | Path = "data") -> tuple[np.ndarray, np.ndarray]:\n    """Return ``(X, y)``: ``X`` float32 (H, W, B) raw radiance, ``y`` int16 (H, W).\n\n    Label 0 means unlabelled; classes are numbered from 1.\n    """\n    if name not in DATASETS:\n        raise KeyError(f"unknown scene {name!r}; known: {sorted(DATASETS)}")\n    root = Path(root)\n    fx, fy = DATASETS[name]\n    X = _first_array(sio.loadmat(root / fx)).astype(np.float32)\n    y = _first_array(sio.loadmat(root / fy)).astype(np.int16)\n    if X.ndim != 3:\n        raise ValueError(f"{name}: expected a 3-D cube, got {X.shape}")\n    if X.shape[:2] != y.shape:\n        raise ValueError(f"{name}: cube {X.shape[:2]} and ground truth {y.shape} disagree")\n    return X, y\n\n\n@dataclass\nclass Split:\n    """Pixel index sets for one scene under one protocol."""\n\n    scene: str\n    protocol: str\n    patch: int\n    seed: int\n    train: np.ndarray  # (N, 2) int32 of (row, col)\n    val: np.ndarray\n    test: np.ndarray\n    split_map: np.ndarray | None = None  # (H, W) int8: 0 train, 1 val, 2 test\n    n_dropped_buffer: int = 0\n    buffer: int = 0\n    block: int = 0\n    classes: np.ndarray = field(default_factory=lambda: np.array([], dtype=np.int16))\n\n    def counts(self) -> dict[str, int]:\n        return {"train": len(self.train), "val": len(self.val), "test": len(self.test)}\n\n\ndef random_split(\n    y: np.ndarray,\n    patch: int,\n    seed: int = 0,\n    train_frac: float = 0.10,\n    val_frac: float = 0.05,\n    scene: str = "",\n) -> Split:\n    """Conventional stratified random split over labelled pixels.\n\n    Patch windows may overlap across splits; this is the protocol whose optimism\n    the study measures.\n    """\n    rng = np.random.default_rng(seed)\n    classes = np.unique(y[y > 0])\n    tr, va, te = [], [], []\n    for c in classes:\n        rr, cc = np.where(y == c)\n        pick = np.stack([rr, cc], 1)[rng.permutation(len(rr))]\n        n_tr = max(1, int(round(train_frac * len(rr))))\n        n_va = max(1, int(round(val_frac * len(rr))))\n        tr.append(pick[:n_tr])\n        va.append(pick[n_tr : n_tr + n_va])\n        te.append(pick[n_tr + n_va :])\n    return Split(\n        scene=scene,\n        protocol="random",\n        patch=patch,\n        seed=seed,\n        train=np.concatenate(tr).astype(np.int32),\n        val=np.concatenate(va).astype(np.int32),\n        test=np.concatenate(te).astype(np.int32),\n        classes=classes,\n    )\n\n\ndef _assign_blocks(\n    n_blocks: int, fracs: tuple[float, float, float], rng: np.random.Generator\n) -> np.ndarray:\n    """Label each block 0 (train), 1 (validation) or 2 (test)."""\n    n_tr = max(1, int(round(fracs[0] * n_blocks)))\n    n_va = max(1, int(round(fracs[1] * n_blocks)))\n    n_te = max(1, n_blocks - n_tr - n_va)\n    labels = np.concatenate(\n        [np.zeros(n_tr, np.int8), np.ones(n_va, np.int8), np.full(n_te, 2, np.int8)]\n    )[:n_blocks]\n    rng.shuffle(labels)\n    return labels\n\n\ndef disjoint_split(\n    y: np.ndarray,\n    patch: int,\n    seed: int = 0,\n    block: int | None = None,\n    fracs: tuple[float, float, float] = (0.25, 0.15, 0.60),\n    min_per_class: int = 5,\n    min_val_per_class: int = 2,\n    max_tries: int = 200,\n    scene: str = "",\n    buffer: int | None = None,\n    fixed_classes: np.ndarray | None = None,\n) -> Split:\n    """Spatially disjoint split with a buffer ring around every split boundary.\n\n    The image is tiled into square blocks of side ``block`` and each block is\n    assigned whole to one split.  A labelled pixel survives only if every cell\n    within ``buffer`` of it belongs to its own split, which makes the patch\n    windows of different splits disjoint by construction.\n\n    ``buffer`` defaults to ``patch // 2``.  Set it to the largest patch radius\n    used anywhere in the study and hold it fixed, so that one split serves every\n    patch size and a spatial-radius sweep is not confounded by a changing test\n    set.\n\n    Block assignment is redrawn until every class has at least ``min_per_class``\n    surviving pixels in both train and test, or ``max_tries`` is reached; the\n    best assignment seen is kept and classes that never met the quota are\n    excluded from ``Split.classes`` and from the index sets.\n    """\n    H, W = y.shape\n    r = patch // 2 if buffer is None else int(buffer)\n    if block is None:\n        # Blocks must be comfortably larger than the buffer ring, or the ring\n        # eats the whole block; they must also be numerous enough to partition.\n        block = max(6 * r + 1, min(H, W) // 8)\n    gh, gw = int(np.ceil(H / block)), int(np.ceil(W / block))\n    n_blocks = gh * gw\n    all_classes = (\n        np.unique(y[y > 0]) if fixed_classes is None else np.asarray(fixed_classes, dtype=np.int16)\n    )\n\n    rr, cc = np.where(y > 0)\n    interior = (rr >= r) & (rr < H - r) & (cc >= r) & (cc < W - r)\n    rr, cc = rr[interior], cc[interior]\n    labels = y[rr, cc]\n\n    best: Split | None = None\n    best_score = -1\n    rng = np.random.default_rng(seed)\n\n    for _ in range(max_tries):\n        blk = _assign_blocks(n_blocks, fracs, rng).reshape(gh, gw)\n        split_map = np.repeat(np.repeat(blk, block, axis=0), block, axis=1)[:H, :W]\n\n        own = split_map[rr, cc]\n        # A window is pure exactly when its minimum equals its maximum, which a\n        # separable sliding filter computes for the whole image at once.\n        size = 2 * r + 1\n        lo = ndimage.minimum_filter(split_map, size=size, mode="nearest")\n        hi = ndimage.maximum_filter(split_map, size=size, mode="nearest")\n        keep = lo[rr, cc] == hi[rr, cc]\n\n        kept_classes = []\n        for c in all_classes:\n            o = own[keep & (labels == c)]\n            if (\n                (o == 0).sum() >= min_per_class\n                and (o == 1).sum() >= min_val_per_class\n                and (o == 2).sum() >= min_per_class\n            ):\n                kept_classes.append(c)\n        if len(kept_classes) > best_score:\n            idx = np.stack([rr, cc], 1)[keep]\n            o = own[keep]\n            best = Split(\n                scene=scene,\n                protocol="disjoint",\n                patch=patch,\n                seed=seed,\n                train=idx[o == 0].astype(np.int32),\n                val=idx[o == 1].astype(np.int32),\n                test=idx[o == 2].astype(np.int32),\n                split_map=split_map.astype(np.int8),\n                n_dropped_buffer=int((~keep).sum()),\n                buffer=r,\n                block=block,\n                classes=np.array(kept_classes, dtype=np.int16),\n            )\n            best_score = len(kept_classes)\n        if best_score == len(all_classes):\n            break\n\n    assert best is not None\n    if fixed_classes is not None:\n        # The class set is pinned so that seeds remain comparable; the search\n        # above still prefers assignments that meet the quota for all of them.\n        best.classes = np.sort(np.asarray(fixed_classes, dtype=np.int16))\n    keep_lab = {int(c) for c in best.classes}\n    for attr in SPLIT_NAMES:\n        idx = getattr(best, attr)\n        if len(idx):\n            lab = y[idx[:, 0], idx[:, 1]]\n            m = np.isin(lab, list(keep_lab))\n            setattr(best, attr, idx[m])\n    return best\n\n\ndef audit_leakage(split: Split, strict: bool = True) -> dict:\n    """Measure how much spatial context test patches share with train/val pixels.\n\n    Under ``disjoint`` every count must be zero.  Under ``random`` the counts\n    quantify the shared context that inflates reported accuracy.\n    """\n    r = split.patch // 2\n    out = {\n        "scene": split.scene,\n        "protocol": split.protocol,\n        "patch": split.patch,\n        "seed": split.seed,\n        "test_patches": int(len(split.test)),\n        "touching_train": 0,\n        "touching_val": 0,\n        "shared_train_pixels": 0,\n    }\n    if len(split.test) == 0 or len(split.train) == 0:\n        return out\n\n    hi_r = int(max(split.train[:, 0].max(), split.test[:, 0].max())) + r + 1\n    hi_c = int(max(split.train[:, 1].max(), split.test[:, 1].max())) + r + 1\n    if len(split.val):\n        hi_r = max(hi_r, int(split.val[:, 0].max()) + r + 1)\n        hi_c = max(hi_c, int(split.val[:, 1].max()) + r + 1)\n    mark = np.zeros((hi_r + 1, hi_c + 1), dtype=np.int8)\n    mark[split.train[:, 0], split.train[:, 1]] = 1\n    if len(split.val):\n        mark[split.val[:, 0], split.val[:, 1]] = 2\n\n    for row, col in split.test:\n        lo_r, lo_c = max(0, row - r), max(0, col - r)\n        win = mark[lo_r : row + r + 1, lo_c : col + r + 1]\n        n_tr = int((win == 1).sum())\n        out["touching_train"] += n_tr > 0\n        out["touching_val"] += int((win == 2).any())\n        out["shared_train_pixels"] += n_tr\n    out["touching_train"] = int(out["touching_train"])\n    out["touching_val"] = int(out["touching_val"])\n    out["frac_touching_train"] = out["touching_train"] / out["test_patches"]\n\n    if strict and split.protocol == "disjoint" and (out["touching_train"] or out["touching_val"]):\n        raise AssertionError(f"disjoint split leaked: {out}")\n    return out\n\n\ndef fit_standardiser(X: np.ndarray, train_idx: np.ndarray) -> tuple[np.ndarray, np.ndarray]:\n    """Per-band mean and standard deviation from training pixels only."""\n    vals = X[train_idx[:, 0], train_idx[:, 1], :]\n    mu = vals.mean(0).astype(np.float32)\n    sd = vals.std(0).astype(np.float32)\n    sd[sd == 0] = 1.0\n    return mu, sd\n\n\ndef apply_standardiser(X: np.ndarray, mu: np.ndarray, sd: np.ndarray) -> np.ndarray:\n    return ((X - mu) / sd).astype(np.float32)\n\n\ndef extract_patches(X: np.ndarray, idx: np.ndarray, patch: int) -> np.ndarray:\n    """Return (N, B, patch, patch) float32 patches centred on ``idx``.\n\n    The cube is reflect-padded so border pixels keep a full window.\n    """\n    r = patch // 2\n    Xp = np.pad(X, ((r, r), (r, r), (0, 0)), mode="reflect")\n    out = np.empty((len(idx), X.shape[2], patch, patch), dtype=np.float32)\n    for i, (row, col) in enumerate(idx):\n        out[i] = np.transpose(Xp[row : row + patch, col : col + patch, :], (2, 0, 1))\n    return out\n\n\ndef labels_of(y: np.ndarray, idx: np.ndarray, classes: np.ndarray) -> np.ndarray:\n    """Map ground-truth labels onto contiguous 0..C-1 indices."""\n    lut = np.full(int(classes.max()) + 1, -1, dtype=np.int64)\n    lut[np.asarray(classes, dtype=np.int64)] = np.arange(len(classes))\n    return lut[y[idx[:, 0], idx[:, 1]]]\n\n\ndef class_table(y: np.ndarray, split: Split) -> np.ndarray:\n    """Rows = classes in ``split.classes``, columns = (train, val, test) counts."""\n    out = np.zeros((len(split.classes), 3), dtype=np.int64)\n    for j, idx in enumerate((split.train, split.val, split.test)):\n        if len(idx) == 0:\n            continue\n        lab = y[idx[:, 0], idx[:, 1]]\n        for i, c in enumerate(split.classes):\n            out[i, j] = int((lab == c).sum())\n    return out\n\n\ndef make_split(y: np.ndarray, protocol: str, patch: int, seed: int, scene: str = "", **kw) -> Split:\n    if protocol == "random":\n        return random_split(y, patch=patch, seed=seed, scene=scene, **kw)\n    if protocol == "disjoint":\n        return disjoint_split(y, patch=patch, seed=seed, scene=scene, **kw)\n    raise ValueError(f"unknown protocol {protocol!r}")\n')
open('src/models.py','w').write('"""Baseline architectures for hyperspectral patch classification.\n\nEvery model takes a patch of shape (B, C_in, P, P) where ``C_in`` is the number\nof spectral components after PCA (PCA is fitted on training pixels only) and\npredicts the label of the centre pixel.\n\nParameter counts and FLOPs are reported by ``model_cost`` so that accuracy can\nbe read against model size, which is the primary axis of this study.\n"""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\n\n\nclass CNN2D(nn.Module):\n    """Plain spatial CNN over PCA components."""\n\n    def __init__(self, c_in: int, n_classes: int, width: int = 32):\n        super().__init__()\n        # The conventional two-stage 2-D CNN used as a baseline in the HSI\n        # literature: two convolution blocks each followed by 2x2 pooling.\n        self.body = nn.Sequential(\n            nn.Conv2d(c_in, width, 3, padding=1),\n            nn.BatchNorm2d(width),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.Conv2d(width, width * 2, 3, padding=1),\n            nn.BatchNorm2d(width * 2),\n            nn.ReLU(inplace=True),\n            nn.MaxPool2d(2),\n            nn.AdaptiveAvgPool2d(1),\n        )\n        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(width * 2, n_classes))\n\n    def forward(self, x):\n        return self.head(self.body(x))\n\n\nclass CNN3D(nn.Module):\n    """Spectral-spatial 3-D CNN."""\n\n    def __init__(self, c_in: int, n_classes: int, patch: int = 11):\n        super().__init__()\n        self.features = nn.Sequential(\n            nn.Conv3d(1, 8, (7, 3, 3)),\n            nn.BatchNorm3d(8),\n            nn.ReLU(inplace=True),\n            nn.Conv3d(8, 16, (5, 3, 3)),\n            nn.BatchNorm3d(16),\n            nn.ReLU(inplace=True),\n            nn.Conv3d(16, 32, (3, 3, 3)),\n            nn.BatchNorm3d(32),\n            nn.ReLU(inplace=True),\n        )\n        with torch.no_grad():\n            n = self.features(torch.zeros(1, 1, c_in, patch, patch)).numel()\n        self.head = nn.Sequential(nn.Flatten(), nn.Linear(n, 256), nn.ReLU(inplace=True),\n                                  nn.Dropout(0.4), nn.Linear(256, n_classes))\n\n    def forward(self, x):\n        return self.head(self.features(x.unsqueeze(1)))\n\n\nclass HybridSN(nn.Module):\n    """Roy et al. (2020) HybridSN: three 3-D convolutions followed by a 2-D one."""\n\n    def __init__(self, c_in: int, n_classes: int, patch: int = 11):\n        super().__init__()\n        self.conv3d = nn.Sequential(\n            nn.Conv3d(1, 8, (7, 3, 3)),\n            nn.ReLU(inplace=True),\n            nn.Conv3d(8, 16, (5, 3, 3)),\n            nn.ReLU(inplace=True),\n            nn.Conv3d(16, 32, (3, 3, 3)),\n            nn.ReLU(inplace=True),\n        )\n        with torch.no_grad():\n            z = self.conv3d(torch.zeros(1, 1, c_in, patch, patch))\n        ch2d = z.shape[1] * z.shape[2]\n        self.conv2d = nn.Sequential(nn.Conv2d(ch2d, 64, 3), nn.ReLU(inplace=True))\n        with torch.no_grad():\n            n = self.conv2d(z.reshape(1, ch2d, z.shape[3], z.shape[4])).numel()\n        self.head = nn.Sequential(\n            nn.Flatten(), nn.Linear(n, 256), nn.ReLU(inplace=True), nn.Dropout(0.4),\n            nn.Linear(256, 128), nn.ReLU(inplace=True), nn.Dropout(0.4),\n            nn.Linear(128, n_classes),\n        )\n\n    def forward(self, x):\n        z = self.conv3d(x.unsqueeze(1))\n        z = z.reshape(z.shape[0], z.shape[1] * z.shape[2], z.shape[3], z.shape[4])\n        return self.head(self.conv2d(z))\n\n\nclass SSFTT(nn.Module):\n    """Spectral-spatial feature tokenisation transformer, in the style of Sun et al. (2022).\n\n    A 3-D convolution compresses the spectral axis, a 2-D convolution produces a\n    feature map, the map is tokenised and passed through transformer encoder\n    layers, and a class token carries the prediction.\n    """\n\n    def __init__(self, c_in: int, n_classes: int, patch: int = 11, dim: int = 64,\n                 depth: int = 2, heads: int = 4):\n        super().__init__()\n        self.conv3d = nn.Sequential(nn.Conv3d(1, 8, (7, 3, 3)), nn.BatchNorm3d(8), nn.ReLU(True))\n        with torch.no_grad():\n            z = self.conv3d(torch.zeros(1, 1, c_in, patch, patch))\n        ch2d = z.shape[1] * z.shape[2]\n        self.conv2d = nn.Sequential(nn.Conv2d(ch2d, dim, 3, padding=1), nn.BatchNorm2d(dim), nn.ReLU(True))\n        n_tok = z.shape[3] * z.shape[4]\n        self.cls = nn.Parameter(torch.zeros(1, 1, dim))\n        self.pos = nn.Parameter(torch.zeros(1, n_tok + 1, dim))\n        layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=0.1,\n                                           batch_first=True, norm_first=True)\n        self.enc = nn.TransformerEncoder(layer, depth)\n        self.norm = nn.LayerNorm(dim)\n        self.head = nn.Linear(dim, n_classes)\n        nn.init.trunc_normal_(self.pos, std=0.02)\n        nn.init.trunc_normal_(self.cls, std=0.02)\n\n    def forward(self, x):\n        z = self.conv3d(x.unsqueeze(1))\n        z = z.reshape(z.shape[0], z.shape[1] * z.shape[2], z.shape[3], z.shape[4])\n        z = self.conv2d(z).flatten(2).transpose(1, 2)\n        z = torch.cat([self.cls.expand(z.shape[0], -1, -1), z], 1) + self.pos\n        return self.head(self.norm(self.enc(z))[:, 0])\n\n\ndef model_cost(model: nn.Module, c_in: int, patch: int) -> dict:\n    """Trainable parameter count and forward FLOPs for one patch."""\n    n_par = sum(p.numel() for p in model.parameters() if p.requires_grad)\n    flops = float("nan")\n    try:\n        from torch.utils.flop_counter import FlopCounterMode\n\n        was_training = model.training\n        model.eval()\n        with torch.no_grad():\n            counter = FlopCounterMode(display=False)\n            with counter:\n                model(torch.zeros(1, c_in, patch, patch))\n            flops = float(counter.get_total_flops())\n        if was_training:\n            model.train()\n    except Exception:\n        pass\n    return {"params": int(n_par), "flops": flops}\n\n\nBASELINES = {\n    "CNN2D": CNN2D,\n    "CNN3D": CNN3D,\n    "HybridSN": HybridSN,\n    "SSFTT": SSFTT,\n}\n\n\ndef build(name: str, c_in: int, n_classes: int, patch: int, **kw) -> nn.Module:\n    cls = BASELINES[name] if name in BASELINES else MODELS[name]\n    if cls is CNN2D:\n        return cls(c_in, n_classes, **kw)\n    return cls(c_in, n_classes, patch=patch, **kw)\n\n\n# Filled in by proposed.py once the study\'s own architecture is registered.\nMODELS: dict[str, type[nn.Module]] = dict(BASELINES)\n')
open('src/proposed.py','w').write('"""ASCN, an asymmetric spectral-context network for hyperspectral classification.\n\nThe design follows from what spatially disjoint evaluation shows: a patch-based\nmodel cannot rely on its neighbourhood carrying the answer, because under a\nleakage-free split the neighbourhood no longer contains training pixels of the\nsame object.  What remains reliable is the spectrum of the pixel being\nclassified, with the neighbourhood acting only as weak context.\n\nASCN therefore spends its capacity asymmetrically:\n\n* the **centre pixel** is encoded at full spectral resolution by a small 1-D\n  convolutional encoder, so absorption features keep their shape;\n* the **neighbourhood** is reduced to ``n_groups`` band averages before any\n  spatial mixing, so context costs a fraction of the centre path;\n* a **gate** decides per sample how much of the context to admit.\n\nThe spatial receptive field is exact and set by ``radius``: each 3x3 depthwise\nconvolution extends it by one pixel, and the centre cell of the resulting map is\nread out.  ``radius=0`` removes the context path and leaves a purely spectral\nclassifier, which is the natural ablation floor.\n"""\n\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass SpectralEncoder(nn.Module):\n    """Small strided 1-D CNN over the wavelength axis of a single spectrum."""\n\n    def __init__(self, dim: int = 64, widths: tuple[int, int] = (16, 32), n_tokens: int = 4):\n        super().__init__()\n        w1, w2 = widths\n        self.n_tokens = n_tokens\n        self.net = nn.Sequential(\n            nn.Conv1d(1, w1, 7, stride=2, padding=3),\n            nn.BatchNorm1d(w1),\n            nn.GELU(),\n            nn.Conv1d(w1, w2, 5, stride=2, padding=2),\n            nn.BatchNorm1d(w2),\n            nn.GELU(),\n            nn.Conv1d(w2, dim, 3, stride=2, padding=1),\n            nn.BatchNorm1d(dim),\n            nn.GELU(),\n            # Pooling to a few tokens rather than one keeps coarse information\n            # about WHERE along the spectrum a feature sits, which a single\n            # global average would discard.\n            nn.AdaptiveAvgPool1d(n_tokens),\n        )\n        self.merge = nn.Linear(dim * n_tokens, dim) if n_tokens > 1 else nn.Identity()\n\n    def forward(self, spec: torch.Tensor) -> torch.Tensor:\n        # spec: (N, L) -> (N, dim)\n        z = self.net(spec.unsqueeze(1))\n        return self.merge(z.flatten(1)) if self.n_tokens > 1 else z.squeeze(-1)\n\n\nclass ContextEncoder(nn.Module):\n    """Band-grouped neighbourhood path with an exact receptive radius."""\n\n    def __init__(self, n_groups: int, dim: int, radius: int):\n        super().__init__()\n        self.radius = radius\n        self.proj = nn.Conv2d(n_groups, dim, 1, bias=False)\n        self.blocks = nn.ModuleList(\n            nn.Sequential(\n                nn.Conv2d(dim, dim, 3, padding=0, groups=dim, bias=False),\n                nn.BatchNorm2d(dim),\n                nn.GELU(),\n                nn.Conv2d(dim, dim, 1, bias=False),\n                nn.BatchNorm2d(dim),\n                nn.GELU(),\n            )\n            for _ in range(radius)\n        )\n\n    def forward(self, grouped: torch.Tensor) -> torch.Tensor:\n        # grouped: (N, G, P, P) -> (N, dim) read at the centre cell\n        z = self.proj(grouped)\n        for blk in self.blocks:\n            z = blk(z)  # each 3x3 depthwise conv shrinks the map by one ring\n        c = z.shape[-1] // 2\n        return z[:, :, c, c]\n\n\nclass ASCN(nn.Module):\n    """Asymmetric Spectral-Context Network.\n\n    Args:\n        c_in: number of input bands (full spectrum, or PCA components).\n        n_classes: number of classes.\n        patch: spatial side of the input patch.\n        dim: embedding width shared by both paths.\n        n_groups: band groups used by the context path.\n        radius: exact spatial receptive radius of the context path.\n        fusion: ``gate``, ``sum`` or ``concat``.\n        symmetric: if true the full spectral encoder is applied to every pixel\n            of the patch instead of the centre only.  This is the costly control\n            condition used in the ablation, not the proposed configuration.\n    """\n\n    def __init__(\n        self,\n        c_in: int,\n        n_classes: int,\n        patch: int = 11,\n        dim: int = 64,\n        n_groups: int = 16,\n        radius: int = 2,\n        fusion: str = "gate",\n        symmetric: bool = False,\n    ):\n        super().__init__()\n        if radius > patch // 2:\n            raise ValueError(f"radius {radius} exceeds patch radius {patch // 2}")\n        self.patch = patch\n        self.dim = dim\n        self.n_groups = min(n_groups, c_in)\n        self.radius = radius\n        self.fusion = fusion\n        self.symmetric = symmetric\n\n        self.spectral = SpectralEncoder(dim)\n        self.context = ContextEncoder(self.n_groups, dim, radius) if radius > 0 else None\n        edges = torch.linspace(0, c_in, self.n_groups + 1).round().long()\n        A = torch.zeros(self.n_groups, c_in)\n        for g in range(self.n_groups):\n            lo, hi = int(edges[g]), max(int(edges[g + 1]), int(edges[g]) + 1)\n            A[g, lo:hi] = 1.0 / (hi - lo)\n        self.register_buffer("group_matrix", A)\n\n        if radius > 0 and fusion == "gate":\n            self.gate = nn.Linear(2 * dim, dim)\n        elif radius > 0 and fusion == "concat":\n            self.mix = nn.Linear(2 * dim, dim)\n        self.norm = nn.LayerNorm(dim)\n        self.head = nn.Linear(dim, n_classes)\n\n    def _group_bands(self, x: torch.Tensor) -> torch.Tensor:\n        # Contiguous band averaging expressed as a fixed 1x1 convolution, which\n        # is both cheaper and clearer than reshaping to pool per pixel.\n        return F.conv2d(x, self.group_matrix[:, :, None, None])\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        n, c, p, _ = x.shape\n        mid = p // 2\n        if self.symmetric:\n            spec = x.permute(0, 2, 3, 1).reshape(n * p * p, c)\n            z_all = self.spectral(spec).reshape(n, p, p, self.dim).permute(0, 3, 1, 2)\n            z_c = z_all[:, :, mid, mid]\n        else:\n            z_c = self.spectral(x[:, :, mid, mid])\n\n        if self.context is None:\n            return self.head(self.norm(z_c))\n\n        # The centre readout of the context path depends only on the ring of\n        # radius `radius`, so cropping to it first is exact and much cheaper.\n        lo, hi = mid - self.radius, mid + self.radius + 1\n        z_s = self.context(self._group_bands(x[:, :, lo:hi, lo:hi]))\n        if self.fusion == "gate":\n            g = torch.sigmoid(self.gate(torch.cat([z_c, z_s], 1)))\n            z = g * z_c + (1.0 - g) * z_s\n        elif self.fusion == "sum":\n            z = z_c + z_s\n        else:\n            z = self.mix(torch.cat([z_c, z_s], 1))\n        return self.head(self.norm(z))\n\n\nclass LSSA(nn.Module):\n    """Lightweight Spectral-Spatial Attention network.\n\n    Design history, stated plainly because the motivating premise did not\n    survive a larger training budget.  Under a 20-epoch budget the\n    spectral-spatial transformer baseline led a spectral SVM and a small 2-D CNN\n    by about three points, which suggested that combining the two paths under\n    attention was what mattered.  At 60 epochs that gap closes: mean overall\n    accuracy over four scenes and five seeds is 0.8426 for the 2-D CNN against\n    0.8328 for the transformer, a difference that is not significant\n    (Wilcoxon p = 0.20).  Under-training had favoured the transformer.\n\n    LSSA is therefore best read as a compression of the transformer baseline\n    rather than as an accuracy claim: it keeps the spectral-spatial attention\n    structure and removes the parameters that are not carrying it.\n\n    * the 3-D spectral stem is kept, since it costs almost nothing;\n    * the dense 2-D projection that dominates the baseline\'s parameter count is\n      replaced by a depthwise-separable pair;\n    * the token grid is pooled before attention, so the sequence is short;\n    * one narrow encoder layer replaces two wide ones.\n\n    Args:\n        c_in: input channels (PCA components).\n        n_classes: number of classes.\n        patch: spatial side of the input patch.\n        dim: token width.\n        depth: number of transformer encoder layers.\n        heads: attention heads.\n        pool: token-grid pooling factor; 1 keeps every spatial cell.\n        separable: use the depthwise-separable projection (the proposed form).\n    """\n\n    def __init__(\n        self,\n        c_in: int,\n        n_classes: int,\n        patch: int = 11,\n        dim: int = 48,\n        depth: int = 1,\n        heads: int = 4,\n        pool: int = 2,\n        separable: bool = True,\n    ):\n        super().__init__()\n        self.conv3d = nn.Sequential(nn.Conv3d(1, 8, (7, 3, 3)), nn.BatchNorm3d(8), nn.GELU())\n        with torch.no_grad():\n            z = self.conv3d(torch.zeros(1, 1, c_in, patch, patch))\n        ch2d = z.shape[1] * z.shape[2]\n        if separable:\n            self.proj = nn.Sequential(\n                nn.Conv2d(ch2d, ch2d, 3, padding=1, groups=ch2d, bias=False),\n                nn.BatchNorm2d(ch2d),\n                nn.GELU(),\n                nn.Conv2d(ch2d, dim, 1, bias=False),\n                nn.BatchNorm2d(dim),\n                nn.GELU(),\n            )\n        else:\n            self.proj = nn.Sequential(\n                nn.Conv2d(ch2d, dim, 3, padding=1), nn.BatchNorm2d(dim), nn.GELU()\n            )\n        self.pool = nn.AvgPool2d(pool) if pool > 1 else nn.Identity()\n        with torch.no_grad():\n            g = self.pool(self.proj(z.reshape(1, ch2d, z.shape[3], z.shape[4])))\n        n_tok = g.shape[2] * g.shape[3]\n\n        self.cls = nn.Parameter(torch.zeros(1, 1, dim))\n        self.pos = nn.Parameter(torch.zeros(1, n_tok + 1, dim))\n        layer = nn.TransformerEncoderLayer(dim, heads, dim * 2, dropout=0.1,\n                                           batch_first=True, norm_first=True,\n                                           activation="gelu")\n        self.enc = nn.TransformerEncoder(layer, depth)\n        self.norm = nn.LayerNorm(dim)\n        self.head = nn.Linear(dim, n_classes)\n        nn.init.trunc_normal_(self.pos, std=0.02)\n        nn.init.trunc_normal_(self.cls, std=0.02)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        z = self.conv3d(x.unsqueeze(1))\n        z = z.reshape(z.shape[0], z.shape[1] * z.shape[2], z.shape[3], z.shape[4])\n        z = self.pool(self.proj(z)).flatten(2).transpose(1, 2)\n        z = torch.cat([self.cls.expand(z.shape[0], -1, -1), z], 1) + self.pos\n        return self.head(self.norm(self.enc(z))[:, 0])\n\n\ndef register() -> None:\n    """Make the study\'s own models visible to ``models.build``."""\n    import models as M\n\n    M.MODELS["ASCN"] = ASCN\n    M.MODELS["LSSA"] = LSSA\n')
open('src/train.py','w').write('"""Training and evaluation harness shared by every model and both protocols."""\n\nfrom __future__ import annotations\n\nimport time\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom sklearn.decomposition import PCA\nfrom sklearn.metrics import cohen_kappa_score\n\nimport hsidata as hd\nimport models as M\n\n\n@dataclass\nclass Prepared:\n    scene: str\n    protocol: str\n    seed: int\n    patch: int\n    n_pca: int\n    Xtr: np.ndarray\n    ytr: np.ndarray\n    Xva: np.ndarray\n    yva: np.ndarray\n    Xte: np.ndarray\n    yte: np.ndarray\n    n_classes: int\n    split: hd.Split\n\n\ndef prepare(\n    cube: tuple[np.ndarray, np.ndarray],\n    scene: str,\n    protocol: str,\n    seed: int,\n    patch: int = 11,\n    n_pca: int = 30,\n    block: int | None = None,\n    buffer: int = 7,\n    n_per_class: int | None = None,\n    fixed_classes=None,\n    n_train: int | None = None,\n    n_val: int | None = None,\n    test_cap: int | None = None,\n) -> Prepared:\n    """Build patch tensors for one (scene, protocol, seed) configuration.\n\n    Standardisation and PCA are both fitted on training pixels only.  When\n    ``n_per_class`` is given, the training set is subsampled to that many pixels\n    per class for the low-label regime; validation and test are untouched.\n    """\n    X, y = cube\n    if protocol == "disjoint":\n        sp = hd.disjoint_split(y, patch=patch, buffer=buffer, seed=seed, block=block, scene=scene,\n                               fixed_classes=fixed_classes)\n        hd.audit_leakage(sp)\n    else:\n        sp = hd.random_split(y, patch=patch, seed=seed, scene=scene)\n        if fixed_classes is not None:\n            # The same class set under both protocols, so the comparison is not\n            # confounded by a different label space.\n            keep = np.asarray(fixed_classes, dtype=np.int16)\n            sp.classes = np.sort(keep)\n            for attr in ("train", "val", "test"):\n                idx = getattr(sp, attr)\n                lab = y[idx[:, 0], idx[:, 1]]\n                setattr(sp, attr, idx[np.isin(lab, keep)])\n        # Match the fitting budget of the disjoint protocol so the comparison\n        # isolates spatial leakage rather than training-set size.\n        rng0 = np.random.default_rng(seed + 777)\n        for attr, target in (("train", n_train), ("val", n_val)):\n            idx = getattr(sp, attr)\n            if target is not None and target < len(idx):\n                setattr(sp, attr, idx[rng0.choice(len(idx), size=target, replace=False)])\n\n    if test_cap is not None and len(sp.test) > test_cap:\n        # Stratified subsample of the test set; this costs precision, not bias.\n        rng1 = np.random.default_rng(seed + 31337)\n        lab = y[sp.test[:, 0], sp.test[:, 1]]\n        share = test_cap / len(sp.test)\n        pick = []\n        for c in np.unique(lab):\n            pos = np.where(lab == c)[0]\n            take = max(1, int(round(share * len(pos))))\n            pick.append(rng1.choice(pos, size=min(take, len(pos)), replace=False))\n        sp.test = sp.test[np.sort(np.concatenate(pick))]\n\n    train_idx = sp.train\n    if n_per_class is not None:\n        rng = np.random.default_rng(seed + 4242)\n        lab = y[train_idx[:, 0], train_idx[:, 1]]\n        pick = []\n        for c in sp.classes:\n            pos = np.where(lab == c)[0]\n            if len(pos) == 0:\n                continue\n            take = min(n_per_class, len(pos))\n            pick.append(rng.choice(pos, size=take, replace=False))\n        train_idx = train_idx[np.concatenate(pick)]\n\n    mu, sd = hd.fit_standardiser(X, train_idx)\n    Xs = hd.apply_standardiser(X, mu, sd)\n\n    if n_pca is None:\n        # Full spectrum: the band axis keeps its wavelength ordering, which a\n        # spectral convolution needs and PCA destroys.\n        Xp = Xs\n    else:\n        flat_train = Xs[train_idx[:, 0], train_idx[:, 1], :]\n        pca = PCA(n_components=min(n_pca, Xs.shape[2]), random_state=seed).fit(flat_train)\n        H, W, _ = Xs.shape\n        Xp = pca.transform(Xs.reshape(-1, Xs.shape[2])).reshape(H, W, -1).astype(np.float32)\n\n    out = {}\n    for name, idx in (("tr", train_idx), ("va", sp.val), ("te", sp.test)):\n        out["X" + name] = hd.extract_patches(Xp, idx, patch)\n        out["y" + name] = hd.labels_of(y, idx, sp.classes)\n    return Prepared(scene=scene, protocol=protocol, seed=seed, patch=patch,\n                    n_pca=Xp.shape[2], n_classes=len(sp.classes), split=sp, **out)\n\n\ndef _loader(X: np.ndarray, y: np.ndarray, bs: int, shuffle: bool):\n    ds = torch.utils.data.TensorDataset(torch.from_numpy(X), torch.from_numpy(y))\n    # A trailing batch of one breaks BatchNorm in training mode, so drop it when\n    # training; evaluation keeps every sample.\n    drop_last = shuffle and len(X) > bs and len(X) % bs == 1\n    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle,\n                                       num_workers=0, drop_last=drop_last)\n\n\ndef metrics(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> dict:\n    oa = float((y_true == y_pred).mean())\n    per = []\n    for c in range(n_classes):\n        m = y_true == c\n        if m.sum():\n            per.append(float((y_pred[m] == c).mean()))\n    return {"OA": oa, "AA": float(np.mean(per)), "kappa": float(cohen_kappa_score(y_true, y_pred))}\n\n\ndef run_torch_model(\n    name: str,\n    data: Prepared,\n    seed: int = 0,\n    epochs: int = 80,\n    batch: int = 128,\n    lr: float = 1e-3,\n    patience: int = 15,\n    model_kw: dict | None = None,\n) -> dict:\n    """Train one model and return metrics, cost and timing."""\n    torch.manual_seed(seed)\n    np.random.seed(seed)\n    model = M.build(name, data.n_pca, data.n_classes, data.patch, **(model_kw or {}))\n    cost = M.model_cost(model, data.n_pca, data.patch)  # counted on CPU\n    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = model.to(dev)\n\n    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)\n    lossf = nn.CrossEntropyLoss()\n    tr = _loader(data.Xtr, data.ytr, batch, True)\n    va = _loader(data.Xva, data.yva, 512, False)\n\n    best_oa, best_state, bad, t0 = -1.0, None, 0, time.time()\n    for _ in range(epochs):\n        model.train()\n        for xb, yb in tr:\n            xb, yb = xb.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)\n            opt.zero_grad()\n            loss = lossf(model(xb), yb)\n            loss.backward()\n            opt.step()\n        sched.step()\n        model.eval()\n        preds = []\n        with torch.no_grad():\n            for xb, _ in va:\n                preds.append(model(xb.to(dev)).argmax(1).cpu().numpy())\n        oa = float((np.concatenate(preds) == data.yva).mean()) if len(data.yva) else 0.0\n        if oa > best_oa:\n            best_oa, bad = oa, 0\n            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}\n        else:\n            bad += 1\n            if bad >= patience:\n                break\n    train_s = time.time() - t0\n    if best_state is not None:\n        model.load_state_dict(best_state)\n\n    model.eval()\n    preds = []\n    t1 = time.time()\n    with torch.no_grad():\n        for xb, _ in _loader(data.Xte, data.yte, 512, False):\n            preds.append(model(xb.to(dev)).argmax(1).cpu().numpy())\n    if dev.type == "cuda":\n        torch.cuda.synchronize()\n    infer_s = time.time() - t1\n    yp = np.concatenate(preds)\n    out = {"model": name, "scene": data.scene, "protocol": data.protocol, "seed": seed,\n           "patch": data.patch, "n_classes": data.n_classes,\n           "n_train": len(data.ytr), "n_test": len(data.yte),\n           "val_OA": best_oa, "train_s": train_s, "infer_s": infer_s,\n           "infer_us_per_patch": 1e6 * infer_s / max(1, len(data.yte)), **cost}\n    out.update(metrics(data.yte, yp, data.n_classes))\n    return out\n\n\ndef run_svm(data: Prepared, seed: int = 0, c_grid=(1, 10, 100), g_grid=("scale",)) -> dict:\n    """Spectral-only RBF SVM on the centre pixel, tuned on the validation split."""\n    from sklearn.svm import SVC\n\n    r = data.patch // 2\n    tr = data.Xtr[:, :, r, r]\n    va = data.Xva[:, :, r, r]\n    te = data.Xte[:, :, r, r]\n    best, best_oa = None, -1.0\n    t0 = time.time()\n    for C in c_grid:\n        for g in g_grid:\n            m = SVC(C=C, gamma=g, kernel="rbf", random_state=seed).fit(tr, data.ytr)\n            oa = float((m.predict(va) == data.yva).mean()) if len(data.yva) else 0.0\n            if oa > best_oa:\n                best, best_oa = m, oa\n    train_s = time.time() - t0\n    t1 = time.time()\n    yp = best.predict(te)\n    infer_s = time.time() - t1\n    out = {"model": "SVM", "scene": data.scene, "protocol": data.protocol, "seed": seed,\n           "patch": data.patch, "n_classes": data.n_classes,\n           "n_train": len(data.ytr), "n_test": len(data.yte),\n           "val_OA": best_oa, "train_s": train_s, "infer_s": infer_s,\n           "infer_us_per_patch": 1e6 * infer_s / max(1, len(data.yte)),\n           "params": int(best.support_vectors_.size), "flops": float("nan")}\n    out.update(metrics(data.yte, yp, data.n_classes))\n    return out\n')
open('src/run_grid.py','w').write('"""Run the protocol cross: every model, scene, protocol and seed.\n\nResults are appended to the output CSV after each run, so a partial grid is\nstill usable if the process is interrupted.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport hsidata as hd  # noqa: E402\nimport proposed  # noqa: E402\nimport train as T  # noqa: E402\n\nproposed.register()\n\n# Models that take the full spectrum rather than the PCA-reduced input.\nFULL_SPECTRUM = {"ASCN"}\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", default="protocol_config.json")\n    ap.add_argument("--out", default="results_protocol_cross.csv")\n    ap.add_argument("--models", default="SVM,CNN2D,CNN3D,HybridSN,SSFTT")\n    ap.add_argument("--scenes", default="")\n    ap.add_argument("--protocols", default="disjoint,random")\n    ap.add_argument("--seeds", default="0,1,2")\n    ap.add_argument("--patch", type=int, default=11)\n    ap.add_argument("--pca", type=int, default=30)\n    ap.add_argument("--epochs", type=int, default=30)\n    ap.add_argument("--patience", type=int, default=7)\n    ap.add_argument("--batch", type=int, default=256)\n    ap.add_argument("--test-cap", type=int, default=20000)\n    ap.add_argument("--threads", type=int, default=8)\n    args = ap.parse_args()\n\n    torch.set_num_threads(args.threads)\n    cfg = json.load(open(args.config))\n    blocks = cfg["bench_blocks"]\n    fixed = {k: np.array(v) for k, v in cfg["fixed_classes"].items()}\n    scenes = args.scenes.split(",") if args.scenes else list(blocks)\n    models = args.models.split(",")\n    protocols = args.protocols.split(",")\n    seeds = [int(s) for s in args.seeds.split(",")]\n\n    done = set()\n    if os.path.exists(args.out):\n        prev = pd.read_csv(args.out)\n        done = {(r.model, r.scene, r.protocol, r.seed) for r in prev.itertuples()}\n        print(f"resuming, {len(done)} runs already present", flush=True)\n\n    for scene in scenes:\n        cube = hd.load_cube(scene)\n        for seed in seeds:\n            # The disjoint split sets the fitting budget that the random\n            # protocol is then matched to.\n            ref = T.prepare(cube, scene, "disjoint", seed=seed, patch=args.patch,\n                            n_pca=args.pca, block=blocks[scene], fixed_classes=fixed[scene],\n                            test_cap=args.test_cap)\n            budget = {"n_train": len(ref.ytr), "n_val": len(ref.yva)}\n            for protocol in protocols:\n                cache: dict[object, T.Prepared] = {}\n                if protocol == "disjoint":\n                    cache[args.pca] = ref\n                for name in models:\n                    key = (name, scene, protocol, seed)\n                    if key in done:\n                        continue\n                    # ASCN reads the full spectrum; every other model takes the\n                    # conventional PCA-reduced input.\n                    want = None if name in FULL_SPECTRUM else args.pca\n                    if want not in cache:\n                        cache[want] = T.prepare(\n                            cube, scene, protocol, seed=seed, patch=args.patch, n_pca=want,\n                            block=blocks[scene], fixed_classes=fixed[scene],\n                            test_cap=args.test_cap,\n                            **({} if protocol == "disjoint" else budget),\n                        )\n                    data = cache[want]\n                    t0 = time.time()\n                    try:\n                        if name == "SVM":\n                            row = T.run_svm(data, seed=seed)\n                        else:\n                            row = T.run_torch_model(name, data, seed=seed, epochs=args.epochs,\n                                                    batch=args.batch, patience=args.patience)\n                    except Exception as exc:  # keep the grid alive\n                        print(f"FAILED {key}: {exc}", flush=True)\n                        continue\n                    row["wall_s"] = time.time() - t0\n                    row["n_pca"] = data.n_pca\n                    pd.DataFrame([row]).to_csv(\n                        args.out, mode="a", header=not os.path.exists(args.out), index=False\n                    )\n                    print(\n                        f"{name:9s} {scene:9s} {protocol:8s} seed{seed} "\n                        f"OA={row[\'OA\']:.4f} AA={row[\'AA\']:.4f} k={row[\'kappa\']:.4f} "\n                        f"({row[\'wall_s\']:.0f}s)",\n                        flush=True,\n                    )\n\n\nif __name__ == "__main__":\n    main()\n')
open('src/run_ablation.py','w').write('"""Ablation study for ASCN.\n\nEach variant changes one design choice at a time against the proposed\nconfiguration (full spectrum, 16 band groups, radius 2, gated fusion).  Results\nare appended after every run so a partial sweep stays usable.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport hsidata as hd  # noqa: E402\nimport proposed  # noqa: E402\nimport train as T  # noqa: E402\n\nproposed.register()\n\nBASE = dict(radius=2, n_groups=16, dim=64, fusion="gate", symmetric=False)\n\n\ndef variants() -> list[tuple[str, str, dict, object]]:\n    """(axis, label, model kwargs, n_pca) for every ablation cell."""\n    out: list[tuple[str, str, dict, object]] = [("base", "proposed", dict(BASE), None)]\n    for r in (0, 1, 3, 4):\n        out.append(("radius", f"radius={r}", {**BASE, "radius": r}, None))\n    for g in (4, 8, 32):\n        out.append(("n_groups", f"groups={g}", {**BASE, "n_groups": g}, None))\n    for f in ("sum", "concat"):\n        out.append(("fusion", f"fusion={f}", {**BASE, "fusion": f}, None))\n    out.append(("spectral_path", "symmetric", {**BASE, "symmetric": True}, None))\n    out.append(("input", "PCA-30", dict(BASE), 30))\n    for d in (32, 128):\n        out.append(("width", f"dim={d}", {**BASE, "dim": d}, None))\n    return out\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", default="protocol_config.json")\n    ap.add_argument("--out", default="results_ablation.csv")\n    ap.add_argument("--scenes", default="")\n    ap.add_argument("--seeds", default="0,1")\n    ap.add_argument("--axes", default="")\n    ap.add_argument("--patch", type=int, default=11)\n    ap.add_argument("--epochs", type=int, default=20)\n    ap.add_argument("--patience", type=int, default=6)\n    ap.add_argument("--batch", type=int, default=256)\n    ap.add_argument("--test-cap", type=int, default=20000)\n    ap.add_argument("--threads", type=int, default=8)\n    args = ap.parse_args()\n\n    torch.set_num_threads(args.threads)\n    cfg = json.load(open(args.config))\n    blocks = cfg["bench_blocks"]\n    fixed = {k: np.array(v) for k, v in cfg["fixed_classes"].items()}\n    scenes = args.scenes.split(",") if args.scenes else list(blocks)\n    seeds = [int(s) for s in args.seeds.split(",")]\n    axes = set(args.axes.split(",")) if args.axes else None\n\n    done = set()\n    if os.path.exists(args.out):\n        prev = pd.read_csv(args.out)\n        done = {(r.label, r.scene, r.seed) for r in prev.itertuples()}\n        print(f"resuming, {len(done)} runs already present", flush=True)\n\n    for scene in scenes:\n        cube = hd.load_cube(scene)\n        for seed in seeds:\n            cache: dict[object, T.Prepared] = {}\n            for axis, label, kw, n_pca in variants():\n                if axes is not None and axis not in axes:\n                    continue\n                if (label, scene, seed) in done:\n                    continue\n                if n_pca not in cache:\n                    cache[n_pca] = T.prepare(\n                        cube, scene, "disjoint", seed=seed, patch=args.patch, n_pca=n_pca,\n                        block=blocks[scene], fixed_classes=fixed[scene], test_cap=args.test_cap,\n                    )\n                data = cache[n_pca]\n                t0 = time.time()\n                try:\n                    row = T.run_torch_model("ASCN", data, seed=seed, epochs=args.epochs,\n                                            batch=args.batch, patience=args.patience,\n                                            model_kw=kw)\n                except Exception as exc:\n                    print(f"FAILED {label} {scene} seed{seed}: {exc}", flush=True)\n                    continue\n                row.update({"axis": axis, "label": label, "wall_s": time.time() - t0,\n                            "n_pca_in": data.n_pca, **{f"kw_{k}": v for k, v in kw.items()}})\n                pd.DataFrame([row]).to_csv(\n                    args.out, mode="a", header=not os.path.exists(args.out), index=False\n                )\n                print(f"{label:12s} {scene:9s} seed{seed} OA={row[\'OA\']:.4f} "\n                      f"AA={row[\'AA\']:.4f} params={row[\'params\']} "\n                      f"MFLOPs={row[\'flops\'] / 1e6:.2f} ({row[\'wall_s\']:.0f}s)", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
open('src/run_mechanism.py','w').write('"""Mechanism experiments behind the measurement result.\n\nA  spatial window sweep crossed with protocol.  The block partition uses a\n   fixed buffer of 7, so every patch size up to 15 is evaluated on the same\n   pixels and the sweep is not confounded by a changing test set.\nB  capacity sweep of the 2-D CNN under the leakage-free protocol.\nC  training-budget sweep, which is what flipped the ranking between the early\n   20-epoch runs and the 60-epoch ones.\nD  low-label regime.\n\nEach experiment appends to its own CSV after every run.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nimport sys\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport hsidata as hd  # noqa: E402\nimport proposed  # noqa: E402\nimport train as T  # noqa: E402\n\nproposed.register()\nFULL_SPECTRUM = {"ASCN"}\n\n# Models whose stacked 3x3 convolutions need room: three 3-D convolutions take\n# a patch of side p down to p-6, so anything below 7 cannot be built.\nNEEDS_ROOM = {"CNN3D", "HybridSN"}\n\n\ndef append(path: str, row: dict) -> None:\n    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)\n\n\ndef done_keys(path: str, cols: list[str]) -> set:\n    if not os.path.exists(path):\n        return set()\n    prev = pd.read_csv(path)\n    return {tuple(r) for r in prev[cols].itertuples(index=False)}\n\n\ndef run_one(name, data, seed, epochs, patience, batch, model_kw=None):\n    if name == "SVM":\n        return T.run_svm(data, seed=seed)\n    return T.run_torch_model(name, data, seed=seed, epochs=epochs, batch=batch,\n                             patience=patience, model_kw=model_kw)\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--config", default="protocol_config.json")\n    ap.add_argument("--only", default="A,B,C,D")\n    ap.add_argument("--seeds", default="0,1,2,3,4")\n    ap.add_argument("--epochs", type=int, default=60)\n    ap.add_argument("--patience", type=int, default=12)\n    ap.add_argument("--batch", type=int, default=256)\n    ap.add_argument("--test-cap", type=int, default=20000)\n    args = ap.parse_args()\n\n    cfg = json.load(open(args.config))\n    blocks = cfg["bench_blocks"]\n    fixed = {k: np.array(v) for k, v in cfg["fixed_classes"].items()}\n    scenes = list(blocks)\n    seeds = [int(s) for s in args.seeds.split(",")]\n    only = set(args.only.split(","))\n    cubes = {s: hd.load_cube(s) for s in scenes}\n\n    def prep(scene, protocol, seed, patch, n_pca, **kw):\n        return T.prepare(cubes[scene], scene, protocol, seed=seed, patch=patch, n_pca=n_pca,\n                         block=blocks[scene], fixed_classes=fixed[scene],\n                         test_cap=args.test_cap, **kw)\n\n    # ---------------- A: window size x protocol ----------------\n    if "A" in only:\n        out = "results_window_sweep.csv"\n        have = done_keys(out, ["model", "scene", "protocol", "seed", "patch"])\n        for scene in scenes:\n            for seed in seeds:\n                for patch in (3, 5, 7, 9, 11, 15):\n                    ref = prep(scene, "disjoint", seed, patch, 30)\n                    budget = {"n_train": len(ref.ytr), "n_val": len(ref.yva)}\n                    for protocol in ("disjoint", "random"):\n                        data = ref if protocol == "disjoint" else prep(\n                            scene, protocol, seed, patch, 30, **budget)\n                        for name in ("SVM", "CNN2D", "SSFTT"):\n                            if (name, scene, protocol, seed, patch) in have:\n                                continue\n                            if name in NEEDS_ROOM and patch < 7:\n                                continue\n                            t0 = time.time()\n                            try:\n                                row = run_one(name, data, seed, args.epochs, args.patience,\n                                              args.batch)\n                            except Exception as exc:\n                                print(f"FAILED A {name} {scene} {protocol} s{seed} p{patch}: {exc}",\n                                      flush=True)\n                                continue\n                            row["wall_s"] = time.time() - t0\n                            append(out, row)\n                            print(f"A {name:6s} {scene:8s} {protocol:8s} s{seed} p{patch:2d} "\n                                  f"OA={row[\'OA\']:.4f}", flush=True)\n\n    # ---------------- B: capacity of the 2-D CNN ----------------\n    if "B" in only:\n        out = "results_capacity.csv"\n        have = done_keys(out, ["scene", "seed", "width"])\n        for scene in scenes:\n            for seed in seeds:\n                data = prep(scene, "disjoint", seed, 11, 30)\n                for width in (8, 16, 32, 64, 128):\n                    if (scene, seed, width) in have:\n                        continue\n                    t0 = time.time()\n                    row = run_one("CNN2D", data, seed, args.epochs, args.patience, args.batch,\n                                  model_kw={"width": width})\n                    row.update(width=width, wall_s=time.time() - t0)\n                    append(out, row)\n                    print(f"B CNN2D w={width:3d} {scene:8s} s{seed} OA={row[\'OA\']:.4f} "\n                          f"params={row[\'params\']}", flush=True)\n\n    # ---------------- C: training budget ----------------\n    if "C" in only:\n        out = "results_budget.csv"\n        have = done_keys(out, ["model", "scene", "seed", "epochs"])\n        for scene in scenes:\n            for seed in seeds:\n                data = prep(scene, "disjoint", seed, 11, 30)\n                for epochs in (10, 20, 40, 60, 100):\n                    for name in ("CNN2D", "SSFTT", "HybridSN"):\n                        if (name, scene, seed, epochs) in have:\n                            continue\n                        t0 = time.time()\n                        row = run_one(name, data, seed, epochs, 10 ** 6, args.batch)\n                        row.update(epochs=epochs, wall_s=time.time() - t0)\n                        append(out, row)\n                        print(f"C {name:9s} ep={epochs:3d} {scene:8s} s{seed} "\n                              f"OA={row[\'OA\']:.4f}", flush=True)\n\n    # ---------------- D: low-label regime ----------------\n    if "D" in only:\n        out = "results_low_label.csv"\n        have = done_keys(out, ["model", "scene", "seed", "n_per_class"])\n        for scene in scenes:\n            for seed in seeds:\n                for n_per_class in (5, 10, 25):\n                    cache = {}\n                    for name in ("SVM", "CNN2D", "SSFTT", "LSSA"):\n                        if (name, scene, seed, n_per_class) in have:\n                            continue\n                        n_pca = None if name in FULL_SPECTRUM else 30\n                        if n_pca not in cache:\n                            cache[n_pca] = prep(scene, "disjoint", seed, 11, n_pca,\n                                                n_per_class=n_per_class)\n                        t0 = time.time()\n                        row = run_one(name, cache[n_pca], seed, args.epochs, args.patience,\n                                      args.batch)\n                        row.update(n_per_class=n_per_class, wall_s=time.time() - t0)\n                        append(out, row)\n                        print(f"D {name:6s} n={n_per_class:2d} {scene:8s} s{seed} "\n                              f"OA={row[\'OA\']:.4f}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')
open('src/make_outputs.py','w').write('"""Build the study\'s result tables and figures from the experiment CSVs.\n\nRun inside the environment that carries the figure-style helpers, since the\nfigures are deliverables.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nMODEL_ORDER = ["SVM", "CNN2D", "CNN3D", "HybridSN", "SSFTT", "ASCN"]\nPRETTY = {\n    "SVM": "spectral SVM",\n    "CNN2D": "2-D CNN",\n    "CNN3D": "3-D CNN",\n    "HybridSN": "HybridSN",\n    "SSFTT": "SSFTT",\n    "ASCN": "ASCN (ours)",\n}\nSCENE_PRETTY = {\n    "PaviaU": "Pavia University",\n    "PaviaC": "Pavia Centre",\n    "Salinas": "Salinas",\n    "Houston": "Houston 2013",\n}\n\n\ndef protocol_cross_table(cross: pd.DataFrame) -> pd.DataFrame:\n    """Mean and spread of OA under each protocol, with the paired drop."""\n    g = cross.groupby(["scene", "model", "protocol"])["OA"]\n    tab = g.agg(["mean", "std", "count"]).reset_index()\n    wide = tab.pivot(index=["scene", "model"], columns="protocol",\n                     values=["mean", "std", "count"])\n    wide.columns = [f"{a}_{b}" for a, b in wide.columns]\n    wide["drop"] = wide["mean_random"] - wide["mean_disjoint"]\n    # Paired by seed, so the per-seed difference carries the uncertainty.\n    pairs = cross.pivot_table(index=["scene", "model", "seed"], columns="protocol",\n                              values="OA").dropna()\n    d = (pairs["random"] - pairs["disjoint"]).groupby(level=[0, 1])\n    wide["drop_sd"] = d.std()\n    wide["drop_min"] = d.min()\n    wide["drop_max"] = d.max()\n    return wide.reset_index()\n\n\ndef cost_table(cross: pd.DataFrame) -> pd.DataFrame:\n    """Accuracy under the leakage-free protocol against model size."""\n    dis = cross[cross.protocol == "disjoint"]\n    out = dis.groupby("model").agg(\n        OA=("OA", "mean"), OA_sd=("OA", "std"),\n        AA=("AA", "mean"), kappa=("kappa", "mean"),\n        params=("params", "median"), flops=("flops", "median"),\n        infer_us=("infer_us_per_patch", "mean"),\n    )\n    out["MFLOPs"] = out["flops"] / 1e6\n    out = out.drop(columns="flops")\n    return out.reindex([m for m in MODEL_ORDER if m in out.index]).reset_index()\n\n\ndef paired_drop_test(cross: pd.DataFrame) -> pd.DataFrame:\n    """Wilcoxon signed-rank test of the random-minus-disjoint difference per model."""\n    from scipy.stats import wilcoxon\n\n    pairs = cross.pivot_table(index=["scene", "model", "seed"], columns="protocol",\n                              values="OA").dropna().reset_index()\n    rows = []\n    for model, g in pairs.groupby("model"):\n        diff = (g["random"] - g["disjoint"]).to_numpy()\n        row = {"model": model, "n_pairs": len(diff), "mean_drop": float(diff.mean()),\n               "median_drop": float(np.median(diff)),\n               "min_drop": float(diff.min()), "max_drop": float(diff.max())}\n        if len(diff) >= 6 and np.ptp(diff) > 0:\n            stat, p = wilcoxon(diff)\n            row["wilcoxon_stat"], row["p_value"] = float(stat), float(p)\n        else:\n            row["wilcoxon_stat"], row["p_value"] = np.nan, np.nan\n        rows.append(row)\n    out = pd.DataFrame(rows)\n    return out.set_index("model").reindex(\n        [m for m in MODEL_ORDER if m in out.model.values]).reset_index()\n\n\ndef ablation_table(abl: pd.DataFrame) -> pd.DataFrame:\n    base = abl[abl.label == "proposed"].groupby("scene")["OA"].mean()\n    g = abl.groupby(["axis", "label"]).agg(\n        OA=("OA", "mean"), OA_sd=("OA", "std"),\n        params=("params", "median"), flops=("flops", "median")).reset_index()\n    g["MFLOPs"] = g["flops"] / 1e6\n    g = g.drop(columns="flops")\n    per_scene = abl.pivot_table(index=["axis", "label"], columns="scene", values="OA")\n    delta = per_scene.sub(base, axis=1).mean(axis=1).rename("delta_vs_proposed")\n    return g.merge(delta.reset_index(), on=["axis", "label"], how="left")\n\n\ndef main(cross_csv="results_protocol_cross.csv", abl_csv="results_ablation.csv") -> dict:\n    import os\n\n    out = {}\n    cross = pd.read_csv(cross_csv)\n    t1 = protocol_cross_table(cross)\n    t1.to_csv("table_protocol_cross.csv", index=False)\n    t2 = cost_table(cross)\n    t2.to_csv("table_cost.csv", index=False)\n    t3 = paired_drop_test(cross)\n    t3.to_csv("table_drop_test.csv", index=False)\n    out.update(protocol_cross=t1, cost=t2, drop_test=t3)\n    if os.path.exists(abl_csv):\n        t4 = ablation_table(pd.read_csv(abl_csv))\n        t4.to_csv("table_ablation.csv", index=False)\n        out["ablation"] = t4\n    return out\n\n\nif __name__ == "__main__":\n    tables = main()\n    for name, t in tables.items():\n        print("===", name)\n        print(t.to_string(index=False))\n')
open('protocol_config.json','w').write('{\n "bench_blocks": {\n  "PaviaU": 33,\n  "PaviaC": 18,\n  "Salinas": 18,\n  "Houston": 33\n },\n "buffer": 7,\n "patch_max": 15,\n "seeds": [\n  0,\n  1,\n  2,\n  3,\n  4\n ],\n "excluded_scenes": [\n  "IndianPines",\n  "KSC",\n  "Botswana"\n ],\n "dropped_classes": {\n  "PaviaU": [],\n  "PaviaC": [],\n  "Salinas": [\n   12,\n   14,\n   16\n  ],\n  "Houston": [\n   3,\n   14\n  ]\n },\n "fixed_classes": {\n  "PaviaU": [\n   1,\n   2,\n   3,\n   4,\n   5,\n   6,\n   7,\n   8,\n   9\n  ],\n  "PaviaC": [\n   1,\n   2,\n   3,\n   4,\n   5,\n   6,\n   7,\n   8,\n   9\n  ],\n  "Salinas": [\n   1,\n   2,\n   3,\n   4,\n   5,\n   6,\n   7,\n   8,\n   9,\n   10,\n   11,\n   13,\n   15\n  ],\n  "Houston": [\n   1,\n   2,\n   4,\n   5,\n   6,\n   7,\n   8,\n   9,\n   10,\n   11,\n   12,\n   13,\n   15\n  ]\n }\n}')
print(sorted(os.listdir('src')))


In [ ]:
# 1. Protocol audit: how much context the conventional split shares
!python -c "import sys; sys.path.insert(0,'src')"
import sys, json, numpy as np, pandas as pd
sys.path.insert(0,'src')
import hsidata as hd
cfg=json.load(open('protocol_config.json'))
rows=[]
for scene, block in cfg['bench_blocks'].items():
    _, y = hd.load_cube(scene)
    for patch in (5,7,9,11,15):
        a=hd.audit_leakage(hd.random_split(y,patch=patch,seed=0,scene=scene))
        rows.append(dict(scene=scene,protocol='random',patch=patch,
                         frac=a['frac_touching_train'],
                         mean_shared=a['shared_train_pixels']/a['test_patches']))
    sp=hd.disjoint_split(y,patch=15,buffer=7,seed=0,block=block,scene=scene,
                         fixed_classes=np.array(cfg['fixed_classes'][scene]))
    a=hd.audit_leakage(sp)
    rows.append(dict(scene=scene,protocol='disjoint',patch=15,frac=a['frac_touching_train'],mean_shared=0.0))
audit=pd.DataFrame(rows); audit.to_csv('results_leakage_audit.csv',index=False)
print(audit.pivot_table(index='patch',columns=['protocol','scene'],values='frac').round(4).to_string())

In [ ]:
# 2. Protocol cross: 7 models x 4 scenes x 2 protocols x 5 seeds
!python src/run_grid.py --out results_protocol_cross_gpu.csv \
    --models SVM,CNN2D,CNN3D,HybridSN,SSFTT,ASCN,LSSA \
    --protocols disjoint,random --seeds 0,1,2,3,4 \
    --epochs 60 --patience 12 --batch 256 --test-cap 20000

In [ ]:
# 3. Mechanism: window size, capacity, training budget, low-label regime
!python src/run_mechanism.py --only A,B,C,D --seeds 0,1,2,3,4 --epochs 60 --patience 12

In [ ]:
# 4. Every table in the paper
!python src/make_paper_tables.py

In [ ]:
# 5. Figures
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({'font.size':8,'axes.spines.top':False,'axes.spines.right':False,
                     'xtick.direction':'out','ytick.direction':'out','savefig.dpi':300})
gpu=pd.read_csv('results_protocol_cross_gpu.csv')
P={'CNN2D':'2-D CNN','SSFTT':'SSFTT','HybridSN':'HybridSN','SVM':'spectral SVM',
   'CNN3D':'3-D CNN','ASCN':'ASCN','LSSA':'LSSA'}
m=gpu.pivot_table(index='model',columns='protocol',values='OA',aggfunc='mean')
cost=gpu[gpu.protocol=='disjoint'].groupby('model').agg(OA=('OA','mean'),sd=('OA','std'),
        params=('params','median'))
order=cost.sort_values('OA',ascending=False).index.tolist()
fig,axes=plt.subplots(1,2,figsize=(7.2,3.2))
ax=axes[0]; y=np.arange(len(order))[::-1]
for i,mod in zip(y,order):
    ax.plot([m.loc[mod,'random'],m.loc[mod,'disjoint']],[i,i],'-',color='0.75',lw=1.4)
ax.scatter([m.loc[k,'random'] for k in order],y,s=26,color='0.45')
ax.scatter([m.loc[k,'disjoint'] for k in order],y,s=30,color='#C2454E')
ax.set_yticks(y); ax.set_yticklabels([P[k] for k in order]); ax.set_xlabel('overall accuracy')
ax=axes[1]
for mod in order:
    f = mod=='CNN2D'
    ax.errorbar(cost.loc[mod,'params'],cost.loc[mod,'OA'],yerr=cost.loc[mod,'sd']/np.sqrt(20),
                fmt='o',ms=7 if f else 5,color='#C2454E' if f else '0.45',capsize=2)
    ax.annotate(P[mod],(cost.loc[mod,'params'],cost.loc[mod,'OA']),xytext=(7,4),
                textcoords='offset points',fontsize=6)
ax.set_xscale('log'); ax.set_xlabel('trainable parameters')
ax.set_ylabel('overall accuracy, leakage-free split')
fig.tight_layout(); fig.savefig('fig_protocol_and_cost.png',bbox_inches='tight')
print('figures written')